# Week 6 Homework: Your Answers vs. Your Vibe-Coded Dashboard

**Week 4 dashboard link:** _(paste your Streamlit URL)_

---

Two weeks ago you vibe-coded a MovieLens dashboard before you had the tools to judge it. Since then you've had the stats lecture (Week 5) and this week's EDA lecture. Now you do it the real way: answer the same 4 questions yourself, by hand, then audit the dashboard against your own work.

**Ground rules**

- 🚫 **No AI for code or answers in Parts 1 and 2.** Lecture notebooks, pandas/seaborn docs, Google, and Stack Overflow are all fair game. Write Part 3 in your own words too.
- 🔒 **Lock in before you peek.** Finish Parts 1 and 2 (answers written down) *before* you open your dashboard. If you look first, you'll anchor on its numbers and the comparison is worthless.
- 🔢 **Show your numbers.** Every answer needs actual values (counts, averages, titles), not just a vibe.
- ✏️ Written answers go in markdown cells (double-click to edit, Shift+Enter to render). Most are 2 to 4 sentences. Short and specific beats long and vague.

## The data

`movie_ratings.csv` has one row per rating: 100,000 ratings from 943 users on 1,682 movies.

- `user_id`, `movie_id`: who rated, which movie
- `rating`: 1 to 5 stars
- `timestamp`, `rating_year`: when the rating was given
- `age`, `gender`, `occupation`, `zip_code`: the rater
- `title`: movie title, with release year in parentheses
- `year`, `decade`: the movie's release year and decade
- `genres`: all of the movie's genres in one string, separated by the pipe character (a movie can have several)

## Setup

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import plotly.express as px

# One line turns every df.plot() into an interactive Plotly chart
pd.options.plotting.backend = "plotly"

# Same category, same color, in every chart
px.defaults.color_discrete_map = {
    "Adelie": "#E07000", "Chinstrap": "#A020F0", "Gentoo": "#00A0A0",  # species
    "MALE": "#2E5EAA", "FEMALE": "#C2185B",                             # sex
}
px.defaults.color_discrete_sequence = ["#5B6B7F"]  # neutral slate for single-series charts

# Readable axis labels everywhere
px.defaults.labels = {
    "culmen_length_mm": "Culmen length (mm)",
    "culmen_depth_mm": "Culmen depth (mm)",
    "flipper_length_mm": "Flipper length (mm)",
    "body_mass_g": "Body mass (g)",
}

# Works whether this notebook sits in the week folder or inside exercise/
data_path = next(p for p in ["data/movie_ratings.csv", "../data/movie_ratings.csv"] if Path(p).exists())
df = pd.read_csv(data_path)
df.head()

,user_id,movie_id,rating,timestamp,age,gender,occupation,zip_code,title,year,decade,genres,rating_year
0,196,242,3,1997-12-04 15:55:49,49,M,writer,55105,Kolya (1996),1996.0,1990.0,Comedy,1997
1,186,302,3,1998-04-04 19:22:22,39,F,executive,00000,L.A. Confidential (1997),1997.0,1990.0,Crime|Film-Noir|Mystery|Thriller,1998
2,22,377,1,1997-11-07 07:18:36,25,M,writer,40206,Heavyweights (1994),1994.0,1990.0,Children|Comedy,1997
3,244,51,2,1997-11-27 05:02:03,28,M,technician,80525,Legends of the Fall (1994),1994.0,1990.0,Drama|Romance|War|Western,1997
4,166,346,1,1998-02-02 05:33:16,47,M,educator,55113,Jackie Brown (1997),1997.0,1990.0,Crime|Drama,1998


---
# Part 1: Methodology

## 1.1 Know your data first

Before answering anything, check what you're working with: shape, column types, missing values, duplicate rows, and the range of the numeric columns.

In [2]:
print("shape:", df.shape)
print("missing values:\n", df.isna().sum(), "\n")
print("duplicate rows:", df.duplicated().sum())
print("duplicate (user, movie) pairs:", df.duplicated(["user_id", "movie_id"]).sum())
print("users:", df.user_id.nunique(), "| movies:", df.movie_id.nunique())
print("rating values:", df.rating.value_counts().sort_index().to_dict())
print("rating_year values:", df.rating_year.value_counts().to_dict())
print("timestamp range:", df.timestamp.min(), "-->", df.timestamp.max())
print("movies with missing year:", df.loc[df.year.isna(), "title"].unique())
df[["rating", "age", "year", "rating_year"]].describe()

shape: (100000, 13)
missing values:
 user_id         0
movie_id        0
rating          0
timestamp       0
age             0
gender          0
occupation      0
zip_code        0
title           0
year           30
decade         30
genres          0
rating_year     0
dtype: int64 

duplicate rows: 0
duplicate (user, movie) pairs: 0
users: 943 | movies: 1682
rating values: {1: 6110, 2: 11370, 3: 27145, 4: 34174, 5: 21201}
rating_year values: {1997: 52899, 1998: 47101}
timestamp range: 1997-09-20 03:05:10 --> 1998-04-22 23:10:38
movies with missing year: <ArrowStringArray>
[                                                      'unknown',
                         'Heidi Fleiss: Hollywood Madam (1995) ',
 'Land Before Time III: The Time of the Great Giving (1995) (V)',
                                           'Two Friends (1986) ',
                    'Marlene Dietrich: Shadow and Light (1996) ']
Length: 5, dtype: str


,rating,age,year,rating_year
count,100000.000000,100000.000000,99970.000000,100000.000000
mean,3.529860,32.969850,1987.748224,1997.471010
std,1.125674,11.562623,14.140014,0.499161
min,1.000000,7.000000,1922.000000,1997.000000
25%,3.000000,24.000000,1986.000000,1997.000000
50%,4.000000,30.000000,1994.000000,1997.000000
75%,4.000000,40.000000,1996.000000,1998.000000
max,5.000000,73.000000,1998.000000,1998.000000


**✏️ Data notes:** The table has 100,000 rows and 13 columns, with 943 users and 1,682 movies. There are no duplicate rows and no user rated the same movie twice. Ratings are whole numbers from 1 to 5 (mean 3.53), and 4 is the most common rating (34,174 of them). The only missing values are `year` and `decade`, which are missing for the same 30 ratings (0.03%). Those belong to a few of the movies like "unknown" and "Heidi Fleiss: Hollywood Madam (1995)". Two things surprised me: `rating_year` only has two values (1997 and 1998) and all timestamps are between Sep 1997 and Apr 1998. Also `genres` is one pipe-separated string, so counting genres needs splitting first. I kept the 30 rows in the data (they're fine for genre and movie questions) and drop them only in Q3, where a release year is required.

## 1.2 Decide before you chart

**Q1 Genre breakdown**
- What you have to decide: What am I counting, ratings or movies? How do I handle movies that have several genres?

- My choice: I decided to Count **distinct movies** per genre. A movie with several genres counts once in each of its genres (explode, then drop duplicate movie/genre pairs).

- Why: The question asks about "the movies that were rated", so the unit is a movie. Counting ratings would just measure which genres are popular with raters (a few blockbusters would dominate). The percentages add up to more than 100% because of multi-genre movies, so I report them as "share of movies that have this genre", not market share. I also show the ratings-based count as a comparison.

**Q2 Genre satisfaction**
- What you have to decide: "Average rating" of what? How do I treat genres that have very little data?

- My choice: Mean of **all ratings** in each genre (explode, so a rating counts for each genre of its movie). I show the number of ratings and movies next to every average, and I flag genres with fewer than 100 ratings as unreliable (only "unknown", with 10 ratings, falls in that group).

- Why: "How much do people like the genre" is the rating-level question. Small genres can easily land at the top or bottom by chance, so I won't call a winner without looking at the count.

**Q3 Ratings over time**
- What you have to decide: Is every year backed by the same amount of data? What do I do with movies that have no release year?

- My choice: Mean rating per release year, using all ratings of movies released that year. I drop the 30 ratings without a year, and I show the number of ratings per year next to the trend. I treat years with fewer than 30 ratings as too thin to trust.

- Why: The data is not as balanced (some years have 2 ratings, 1996 has 18,744), so the trend can't be equally believable everywhere. A missing year can't be placed on a timeline, and 30 rows out of 100,000 won't change anything.

**Q4 Best movies**
- What you have to decide: What is a movie's rating (the mean of what)? Is "at least 50" ≥ or >? What if there's a tie?

- My choice: A movie's rating is the mean of all of its ratings. "At least 50" means **≥ 50**. I apply the floor first, then rank by mean. Ties are broken by number of ratings (more ratings first).

- Why: "At least" literally means ≥. Filtering first means a movie with 3 five-star ratings can't make the list. Breaking ties by count prefers the better-supported movie.

---
# Part 2: Results (by hand, no AI)

Answer each question with a chart or table **and** a written answer with real numbers. Pick the chart type yourself and be ready to defend it in Part 3. Add as many code cells as you need.

## Q1. Genre breakdown

**What's the distribution of genres among the movies that were rated?**

<details>
<summary>Stuck on movies with several genres? Hint</summary>

`genres` is one string like `Crime|Film-Noir|Mystery|Thriller`. To get one row per genre:

```python
genre_df = df.assign(genre=df['genres'].str.split('|')).explode('genre')
```

Heads up: after this, a movie with 3 genres takes up 3 rows. Think about what that does to anything you count or average. Exploding is fine for "how often does a genre show up", but be careful calling the result "market share" (the percentages add up to more than 100%).
</details>

In [3]:
# Q1: your code here
genre_df = df.assign(genre=df["genres"].str.split("|")).explode("genre")
movie_genre = genre_df.drop_duplicates(["movie_id", "genre"])
n_movies = df["movie_id"].nunique()
q1 = (movie_genre.groupby("genre").size().rename("movies").sort_values(ascending=False).to_frame())
q1["pct_of_movies"] = (q1["movies"] / n_movies * 100).round(1)
q1["ratings"] = genre_df.groupby("genre").size()
q1["pct_of_ratings"] = (q1["ratings"] / len(df) * 100).round(1)
print("Total movies:", n_movies, "| movie-genre pairs:", len(movie_genre),
"| percentages sum to", q1["pct_of_movies"].sum().round(1), "% (multi-genre movies)")
display(q1)


Total movies: 1682 | movie-genre pairs: 2893 | percentages sum to 171.9 % (multi-genre movies)


,movies,pct_of_movies,ratings,pct_of_ratings
genre,,,,
Drama,725,43.1,39895,39.9
Comedy,505,30.0,29832,29.8
Action,251,14.9,25589,25.6
Thriller,251,14.9,21872,21.9
Romance,247,14.7,19461,19.5
Adventure,135,8.0,13753,13.8
Children,122,7.3,7182,7.2
Crime,109,6.5,8055,8.1
Sci-Fi,101,6.0,12730,12.7


In [4]:
fig = px.bar(q1.reset_index(), 
             x="movies", 
             y="genre",
             text="movies",
             hover_data=["pct_of_movies", "ratings"],
             title="Genre breakdown of the 1,682 rated movies (a movie can have several genres)",
             labels={"movies": "Number of rated movies with this genre", "genre": ""})
fig.update_yaxes(categoryorder="total ascending")
fig.update_layout(height=600)
fig.show()

**✏️ My answer (with numbers):** _(Which genres dominate, by how much, and what exactly did you count?)_
Answer: I counted distinct movies per genre, and a movie with many genres counted once for each.
**DRAMA** domintaed with 725 of the 1,682 movies (43.1%), followed by **Comedy** with 505 (30.0%). Then come Action and Thriller (251 each, 14.9%) and Romance (247, 14.7%). The 19 genre labels add up to 212,595 ratings-rows for 100,000 ratings, so the percentages sum to far more than 100% and are "share of movies with this genre", not market share. The ranking by ratings looks similar (Drama 39,895, Comedy 29,832, Action 25,589), but Action moves ahead of Thriller there because action movies get more ratings per movie.

## Q2. Genre satisfaction

**Which genres have the highest average rating? Which have the lowest?**

Show how much data is behind each average (for example, the count next to it). An average of 10 ratings and an average of 10,000 ratings are not the same kind of evidence.

In [5]:
# Q2: your code here
q2 = genre_df.groupby("genre")["rating"].agg(ratings="size", mean_rating="mean")
q2["movies"] = movie_genre.groupby("genre").size()
q2["mean_rating"] = q2["mean_rating"].round(3)
q2 = q2.sort_values("mean_rating", ascending=False)
q2["enough_data"] = q2["ratings"] >= 100
display(q2)

,ratings,mean_rating,movies,enough_data
genre,,,,
Film-Noir,1733,3.922,24,True
War,9398,3.816,71,True
Drama,39895,3.687,725,True
Documentary,758,3.673,50,True
Mystery,5245,3.638,61,True
Crime,8055,3.632,109,True
Romance,19461,3.622,247,True
Western,1854,3.613,27,True
Animation,3605,3.577,42,True


In [6]:
plot = q2[q2["enough_data"]].reset_index()
fig = px.bar(plot, 
             x="mean_rating", 
             y="genre",
             text="mean_rating",
             hover_data=["ratings", "movies"], 
             color="ratings", 
             color_continuous_scale="Teal",
             title="Mean rating by genre (>= 100 ratings; 'unknown', n=10, left out)")
fig.update_yaxes(categoryorder="total ascending")
fig.update_xaxes(range=[0, 5])
fig.add_vline(x=df["rating"].mean(), line_dash="dash", line_color="red")
fig.update_layout(height=600)
fig.show()

**✏️ My answer (with numbers):** _(Top and bottom genres, their averages, and how much data each one rests on.)_

Highest: **Film-Noir** (3.92, 1,733 ratings, 24 movies), **War** (3.82, 9,398 ratings, 71 movies), then **Drama** (3.69, 39,895 ratings) and **Documentary** (3.67, only 758 ratings, 50 movies). Lowest: **Fantasy** (3.22, 1,352 ratings, 22 movies), **Horror** (3.29, 5,317 ratings, 92 movies) and **Children** (3.35, 7,182 ratings, 122 movies). "Unknown" has a mean of 3.20, but that rests on 10 ratings of 2 movies, so I ignore it. The overall mean is 3.53, and the whole spread between best and worst genre is only about 0.7 stars, so the differences are small. Film-Noir and Fantasy are the smallest of the real genres, so those two ends rest on about 20 movies each, much less evidence than Drama's 725 movies.

## Q3. Ratings over time

**How has the mean rating changed across movie release years?**

Show how many ratings each year is based on, alongside the trend.

In [7]:
# Q3: your code here
q3 = (df.dropna(subset=["year"]).groupby("year")["rating"].agg(ratings="size", mean_rating="mean").reset_index())
q3["year"] = q3["year"].astype(int)
q3["reliable"] = (q3["ratings"] >= 30).map({True: "30+ ratings", False: "fewer than 30 (unreliable)"})
print("years:", len(q3), "| years with < 30 ratings:", (q3["ratings"] < 30).sum(),"| years with < 100 ratings:", (q3["ratings"] < 100).sum())

years: 71 | years with < 30 ratings: 4 | years with < 100 ratings: 13


In [8]:
fig = px.scatter(q3, 
                 x="year", 
                 y="mean_rating", 
                 size="ratings", 
                 color="reliable",
                 color_discrete_map={"30+ ratings": "Teal", "fewer than 30 (unreliable)": "red"},
                 hover_data=["ratings"], 
                 title="Mean rating by movie release year (dot size = number of ratings)",
                 labels={"year": "Movie release year", "mean_rating": "Mean rating"})
fig.add_scatter(x=q3["year"], y=q3["mean_rating"], mode="lines", line=dict(color="lightgray"), showlegend=False)
fig.add_hline(y=df["rating"].mean(), line_dash="dash", line_color="gray", annotation_text="overall mean")
fig.update_yaxes(range=[2.5, 5])
fig.show()

In [9]:
px.bar(q3,
       x="year", 
       y="ratings", 
       title="Number of ratings per release year",
       labels={"year": "Movie release year", "ratings": "Number of ratings"}).show()

In [10]:
display(q3.set_index("year").loc[[1939, 1942, 1960, 1980, 1990, 1994, 1996, 1997, 1998]])
old, new = df[df["year"] < 1970]["rating"], df[df["year"] >= 1990]["rating"]
print(f"movies before 1970: {len(old):,} ratings, mean {old.mean():.2f} | movies 1990+: {len(new):,} ratings, mean {new.mean():.2f}")

,ratings,mean_rating,reliable
year,,,
1939,573,3.993019,30+ ratings
1942,261,4.398467,30+ ratings
1960,375,3.909333,30+ ratings
1980,1114,3.874327,30+ ratings
1990,2092,3.576482,30+ ratings
1994,9034,3.420080,30+ ratings
1996,18744,3.311086,30+ ratings
1997,15384,3.312467,30+ ratings
1998,851,3.297297,30+ ratings


movies before 1970: 11,237 ratings, mean 3.93 | movies 1990+: 69,452 ratings, mean 3.39


**✏️ My answer (with numbers):** _(What's the trend? Is it equally believable across the whole timeline?)_

The mean rating goes **down** as release year gets later. Movies from before 1970 average 3.93 (11,237 ratings), while movies from 1990 onward average 3.39 (many more ratings). The recent years are steady at about 3.3 to 3.4: 1996 is 3.31 (18,744 ratings), 1997 is 3.31 and 1998 is 3.30. The old end is not equally believable. The early years are thin and noisy: 1926 has 2 ratings, 1930 has 18, 1932 has 12 (4 years have fewer than 30 ratings, 13 have fewer than 100), and the means jump around a lot (for example 1942 is 4.40 on 261 ratings). I'd trust the trend from about 1960 on much more than the individual old years. The overall downward direction holds, but a single old year doesn't mean much on its own.

## Q4. Best movies, with a floor

**What are the 5 best-rated movies that have at least 50 ratings? What changes if you raise the floor to 150?**

In [11]:
# Q4: your code here
movie_stats = (df.groupby(["movie_id", "title"])["rating"].agg(n_ratings="size", mean_rating="mean").reset_index())

def top5(floor):
    return (movie_stats[movie_stats["n_ratings"] >= floor].sort_values(["mean_rating", "n_ratings"], ascending=False).head(5).reset_index(drop=True))

t50, t150 = top5(50), top5(150)
print("movies with >= 50 ratings:", (movie_stats.n_ratings >= 50).sum(), "| >= 150:", (movie_stats.n_ratings >= 150).sum())
print("Floor 50:") 
display(t50.round(3))
print("Floor 150:") 
display(t150.round(3))
print("Dropped out:", set(t50.title) - set(t150.title))
print("Moved in:   ", set(t150.title) - set(t50.title))

both = pd.concat([t50.assign(floor="at least 50 ratings"), t150.assign(floor="at least 150 ratings")])
display(both)


movies with >= 50 ratings: 603 | >= 150: 203
Floor 50:


,movie_id,title,n_ratings,mean_rating
0,408,"Close Shave, A (1995)",112,4.491
1,318,Schindler's List (1993),298,4.466
2,169,"Wrong Trousers, The (1993)",118,4.466
3,483,Casablanca (1942),243,4.457
4,114,Wallace & Gromit: The Best of Aardman Animatio...,67,4.448


Floor 150:


,movie_id,title,n_ratings,mean_rating
0,318,Schindler's List (1993),298,4.466
1,483,Casablanca (1942),243,4.457
2,64,"Shawshank Redemption, The (1994)",283,4.445
3,603,Rear Window (1954),209,4.388
4,12,"Usual Suspects, The (1995)",267,4.386


Dropped out: {'Wrong Trousers, The (1993)', 'Wallace & Gromit: The Best of Aardman Animation (1996)', 'Close Shave, A (1995)'}
Moved in:    {'Rear Window (1954)', 'Usual Suspects, The (1995)', 'Shawshank Redemption, The (1994)'}


,movie_id,title,n_ratings,mean_rating,floor
0,408,"Close Shave, A (1995)",112,4.491071,at least 50 ratings
1,318,Schindler's List (1993),298,4.466443,at least 50 ratings
2,169,"Wrong Trousers, The (1993)",118,4.466102,at least 50 ratings
3,483,Casablanca (1942),243,4.456790,at least 50 ratings
4,114,Wallace & Gromit: The Best of Aardman Animatio...,67,4.447761,at least 50 ratings
0,318,Schindler's List (1993),298,4.466443,at least 150 ratings
1,483,Casablanca (1942),243,4.456790,at least 150 ratings
2,64,"Shawshank Redemption, The (1994)",283,4.445230,at least 150 ratings
3,603,Rear Window (1954),209,4.387560,at least 150 ratings
4,12,"Usual Suspects, The (1995)",267,4.385768,at least 150 ratings


In [12]:
fig = px.bar(both, 
             x="mean_rating", 
             y="title",
             facet_row="floor",
             text="mean_rating", 
             hover_data=["n_ratings"], 
             title="Top 5 movies by mean rating, two floors",
             labels={"mean_rating": "Mean rating", "title": ""})

fig.update_traces(texttemplate="%{x:.2f}")
fig.update_xaxes(range=[0, 5])
fig.update_yaxes(categoryorder="total ascending", matches=None, showticklabels=True)
fig.update_layout(height=600)
fig.show()

**✏️ My answer (with numbers):** _(Both top-5 lists. Which movies dropped out or moved in when the floor went up, and what does that say about small samples?)_

1. A Close Shave (1995) 4.49, 2. Schindler's List (1993) 4.47, 3. The Wrong Trousers (1993) 4.47, 4. Casablanca (1942) 4.46, 5. Wallace & Gromit: The Best of Aardman Animation (1996) 4.45. **Floor 150 (203 movies qualify):** 1. Schindler's List 4.47 (298 ratings), 2. Casablanca 4.46 (243), 3. The Shawshank Redemption (1994) 4.45 (283), 4. Rear Window (1954) 4.39 (209), 5. The Usual Suspects (1995) 4.39 (267). Three movies dropped out (A Close Shave, The Wrong Trousers, Wallace & Gromit: The Best of Aardman Animation), because they have between 50 and 150 ratings. Three moved in (Shawshank, Rear Window, The Usual Suspects). Only Schindler's List and Casablanca stay in both lists. The movies with fewer ratings can sit at the top because a small sample is noisier and a few 5-star raters move the mean a lot. The 150 floor leaves only well-known movies, so I trust that list more. A low floor gives a more "surprising" list, but a less reliable one.

> 🔒 **Stop here.** Parts 1 and 2 should be completely filled in before you open your dashboard.

---
# Part 3: Analysis

Now open your Week 4 dashboard.

## 3.1 How the dashboard got built

Paste 2 prompts that shaped it (from your build log, or from your AI chat history if you didn't keep a log; don't invent any). For each one: what did you ask, what did the AI give you first, and what did you change or accept without checking?


**Prompt 1** (build log, Moment 2: the design rewrite)
- What I asked: Instead of an OK on the data report, I sent a "DESIGN REQUIREMENT (replaces all earlier design notes)". The goal was "a dashboard that looks like nobody else's". I asked for 4 very different design concepts, each scored 1-5 on how likely other students' AIs were to produce it, on data-honesty and on fit. I said to wait for my pick. I also banned the usual defaults (plotly_white, blue/teal, numbered headers, `st.metric` cards, "Source:" captions, cream + serif + red, navy + neon) and asked for one custom Plotly template and a different chart form per question.

- AI's first attempt: Four concepts: A. Riso Zine, B. Freight Manifest, C. Tarot Reading and D. Terrazzo Snack Bar. It recommended A (Riso Zine). Its first build had four chart forms: sorted bars, a dot plot around the overall mean, an annotated line with a vote-count strip, and a rank-change board.

- What I changed (or accepted without checking): I accepted the AI's recommendation as is ("go with your recommendation"). I did not weigh the other three concepts myself, so that choice was mostly the AI's. The self-check in a real browser then caught problems that had passed its automated tests: the Plotly text was not in the zine font, there was a grey slab behind every chart, and the legend collided with an axis title. They were fixed one by one. I left the cramped genre multiselect as it was, because it still works.

**Prompt 2** (build log, Moment 4: bonus visuals)
- What I asked: "Add bonus visuals. Keep the zine concept and voice." The four required charts were not to change. The AI was to look at the whole dataset (age, gender, occupation, decade, rating_year), show 6 candidate ideas, and build the 3 favourites in a separate section labelled bonus. The prompt ended mid-sentence ("...use expanders or"), which was my mistake.

- AI's first attempt: It profiled the data first. Findings included 74% of ratings coming from men, and `rating_year` covering only 1997-98. It then proposed 6 ideas: genre x decade heatmap, divisive movies, popularity vs quality, toughest critics by occupation, genre co-occurrence and an age x decade ribbon. It built 3 of them (divisive movies, popularity vs quality, toughest critics). The first draft of the divisive-movies chart had a wrong headline. It compared "share of 1 or 5 stars", and the quiet film (*Vertigo*) scored higher than the divisive one because its extremes were all fives.

- What I changed (or accepted without checking): I said "do your fav 3", so I accepted its choice of visuals. I never answered the cut-off sentence, so the AI guessed (cached tables and an `st.expander` per chart). The AI's own review caught the bad headline. It was changed to compare 1-2 star votes with 4-5 star votes. The review also fixed the confusing 3-star labels and a "Healthcares" grammar error. It added a note that the toughest-critics ranking rests on only 16 viewers. I did not check the "divisive" ranking myself. It uses standard deviation, which is one defensible definition and not the only one.


## 3.2 Compare the answers

Read the answer off your dashboard for each question and put it next to yours.

**Q1**
- Dashboard says: Drama 724, Comedy 505, Thriller and Action 251 each, Romance 247. The total is 1,676 films, and "Drama and Comedy tag 68% of the films"
- I say: Drama 725, Comedy 505, Action and Thriller 251, Romance 247, out of 1,682 movies.
- Same? (yes / partly / no): Partly. The ranking is identical, but four genre counts and the total are slightly off.
- If different, why?: See the explanation below.

**Q2**
- Dashboard says: Film-Noir 3.92 is highest and Fantasy 3.22 is lowest. The next lowest are Horror 3.29 and Children 3.35. The overall mean is 3.53.
- I say: the same top and bottom genres. Documentary comes out at 3.67, just below Drama at 3.69.
- Same? (yes / partly / no): Yes for the top and bottom. Partly for the middle, where Documentary and Drama swap places.
- If different, why? See the explanation below.

**Q3**
- Dashboard says: newer films score lower, about 0.14★ less per decade. The high is 1942 at 4.40 and the low is 1998 at 3.30.
- I say: movies before 1970 average 3.93 and movies from 1990 on average 3.39. 1942 is 4.40 and 1998 is 3.30.
- Same? (yes / partly / no): Yes 
- If different, why? There's no real difference. The dashboard says "10 years under 30 votes" while I counted 4. That's because the dashboard also counts the 6 years in 1922–1998 with no films at all. I checked this: 77 years in the span, 71 with data, and 4 of those under 30 ratings, so 6 + 4 = 10.

**Q4**
- Dashboard says: floor 50 gives A Close Shave 4.49, Schindler's List 4.47, The Wrong Trousers 4.47, Casablanca 4.46 and Wallace & Gromit 4.45. Floor 150 gives Schindler's List, Casablanca, Shawshank 4.45, Rear Window 4.39 and The Usual Suspects 4.39. It says "2 of the top 5 survive".
- I say: the same two lists, with the same ratings and vote counts.
- Same? (yes / partly / no): Yes
- If different, why? There's nothing to explain. I trust it because both of us used ≥ for the floor, ranked by mean with ties going to the movie with more votes, and applied the floor before taking the top 5. The 5 movies with missing years aren't in either top 5, so dropping them changes nothing here.

**When numbers differ, check these first.** Most mismatches come from one of these:

- Counting **ratings** vs. counting **movies**
- One row per rating vs. **one row per rating per genre** (after exploding)
- **Mean of all ratings** vs. the mean of each movie's average
- "At least 50" as ≥ vs. >, or applying the floor before vs. after another step
- What happened to movies with a **missing year**
- Genres with **tiny sample sizes** left in or filtered out

**✏️ Pick two questions and explain what's going on** (2 to 3 sentences each). Track down the *actual cause*: read your dashboard's code, or re-run it with one change. "The AI did it differently" is not an explanation. If the answers match, say why you now trust them.



**Two questions explained**

**Q1** (the counts are off by 1 or 2). The dashboard drops the 30 ratings with no release year when it loads the data, because every chart there depends on year. Those 30 ratings belong to 5 movies. Re-running my count without them gives exactly the dashboard's numbers: Drama 725→724, Children 122→121, Animation 42→41, Documentary 50→48 (Heidi Fleiss and Marlene Dietrich), and "unknown" 2→1. The total goes from 1,682 to 1,676 because the 5 missing-year movies are removed and the remaining "unknown" movie is excluded from the genre charts. So the cause is what happened to the missing-year movies, which I kept for Q1 and the dashboard dropped. Neither choice is wrong, but mine matches the question "movies that were rated". The "68%" is not Drama + Comedy added up. That sum would be 73.3%. I checked, and it's the share of films tagged Drama or Comedy (or both), 68.0% of 1,676.

**Q2** (Documentary and Drama swap places). My Documentary mean is 3.673 on 758 ratings. The dashboard has 3.688 on 744 ratings, because the 14 ratings of Heidi Fleiss and Marlene Dietrich have no year and were dropped. That lifts Documentary just above Drama (3.687), so they look tied at 3.69 and the order flips. The cause is the missing-year rows again, not a different averaging method. Both of us average over all ratings. The dashboard's minimum-ratings guard is 500 and mine is 100, but that makes no difference here, because every one of the 18 real genres clears both. Only "unknown" is removed by my 100 guard. I trust the top and bottom answers because Film-Noir 3.922, Fantasy 3.215 and Horror 3.290 are identical in both versions.

## 3.3 Chart choices

**Q1**
- Chart the dashboard used: a sorted horizontal bar chart. The top two bars (Drama and Comedy) are in violet and the rest in yellow. It has count labels on the bars, a takeaway headline, and a note that the bars add up to more than the number of films.

- Chart I used: a sorted horizontal bar chart (px.bar) with count labels. The percentage of movies and the ratings count are only in the hover. The title says a movie can have several genres.

- Which communicates the answer better, and why?:  It's close to a tie, and the form is right for both: 18 categories with one number each, sorted, readable labels, and an axis that starts at zero. I think the dashboard is a little better, because the highlight and headline show "Drama and Comedy dominate" at a glance. Mine is better at showing the percentage and rating counts, though only on hover. I would keep the same form for both.

**Q2**
- Chart the dashboard used: a dot plot ("lollipop") around the overall mean (3.53). Violet dots are above it and pink dots below, with a "zoomed axis" warning in the text.

- Chart I used: a sorted horizontal bar chart with the x-axis from 0 to 5, coloured by number of ratings, with a red dashed line at the overall mean.

- Which communicates the answer better, and why?  The dashboard's, for the question "highest and lowest". With a 0–5 axis my bars all look almost the same length, because the whole spread is only 0.7 stars. The dashboard lines every genre up against the overall mean and uses two colours for above and below. The zoomed axis would be misleading on bars, but it's fair here because the stems start at the mean, not at zero, and the text says so. Mine is better on honesty about scale, because it doesn't exaggerate, and on showing how much data is behind each number (the colour and hover). However, the colour scale is hard to read. As far as the screenshot shows, the dashboard doesn't show counts on the chart itself.

**Q3**
- Chart the dashboard used: a line of mean rating by release year with a dashed trend line weighted by votes. Years with fewer than 30 votes are left blank. The highest and lowest points are labelled, and a bar strip of vote counts sits under the line on the same x-axis. The y-axis is zoomed and says so.

- Chart I used: a bubble scatter where dot size is the number of ratings. Years with fewer than 30 ratings are red. A grey line joins the dots, and there's a dashed line at the overall mean. The y-axis runs from 2.5 to 5. A separate bar chart shows ratings per year.

- Which communicates the answer better, and why?: The dashboard's. A line is the right form for a trend over time, and the vote-count strip lined up underneath shows right away that the old years rest on very little data and the 1990s on a lot. Bubble size is hard to compare, and my count chart is a separate figure to look at. The dashboard also hides thin years instead of drawing them. Both charts cut the y-axis, which exaggerates the slope, but the dashboard says so in the text and mine doesn't. One weakness of the dashboard is that it labels 1942 as "high" even though that year is a single 261-rating spike among noisy years.

**Q4**
- Chart the dashboard used:  a ranked leaderboard (bump or slope chart) from the 50-vote list to the 150-vote list. Each line shows a film moving up, and a pink X shows a film that doesn't clear 150 votes. Rating and vote count are in each label.

- Chart I used: two stacked panels of bars (facet_row) with an x-axis from 0 to 5. The rating is on the bar and the vote count is only in the hover.

- Which communicates the answer better, and why?: The dashboard's. The question asks what changes when the floor goes up, and the lines show who dropped out, who moved in and who stayed, in one picture. My bars all look identical (4.39–4.49 on a 0–5 axis), so the ranking has to be read from the numbers, and the vote counts that explain the change are hidden in hover. The dashboard's weaknesses are that some long titles are cut off ("Wallace & Gromit: The Best of A…") and the crossing lines are busy.

**Overall**. The dashboard's forms are better for Q2, Q3 and Q4, because they put the comparison the question asks for into the shape of the chart. Mine are plainer and safer on axes, and they hide the sample sizes in hover. For Q1 the two charts are about equal.

Use what you learned about chart choice and misleading charts (Week 5) and this week's lecture. Is it sorted? Are the labels readable? Are the axes honest? Does it show how much data is behind each number?

In [13]:
users = df.drop_duplicates("user_id")
per_user = df.groupby("user_id").size()
print("ratings per user: min", per_user.min(), "| median", per_user.median(), "| max", per_user.max())
print("rating_year values:", df.rating_year.unique())
print("users: median age", users.age.median(), "| share men", round((users.gender == "M").mean(), 2))

ratings per user: min 20 | median 65.0 | max 737
rating_year values: [1997 1998]
users: median age 31.0 | share men 0.71


In [14]:
def ci(genre):
    s = genre_df.loc[genre_df.genre == genre, "rating"]
    half = 1.96 * s.std() / len(s) ** 0.5
    return len(s), s.mean(), s.mean() - half, s.mean() + half
for g in ["Fantasy", "Horror"]:
    n, m, lo, hi = ci(g)
    print(f"{g}: {n} ratings, mean {m:.3f}, 95% CI [{lo:.3f}, {hi:.3f}]")
fan = genre_df[genre_df.genre == "Fantasy"]
print("Fantasy movies:", fan.movie_id.nunique())
print("top 3 movies' share:", round(fan.groupby("title").size().nlargest(3).sum() / len(fan) * 100), "%")

Fantasy: 1352 ratings, mean 3.215, 95% CI [3.156, 3.275]
Horror: 5317 ratings, mean 3.290, 95% CI [3.258, 3.322]
Fantasy movies: 22
top 3 movies' share: 46 %


In [15]:
print("\nunique rating_year:", df.rating_year.nunique())
movies = df.dropna(subset=["year"]).groupby("movie_id").agg(year=("year", "first"), mean=("rating", "mean"))
print("movies before 1970:", (movies.year < 1970).sum(), "of", len(movies))
print("avg of movie means: before 1970 =", round(movies[movies.year < 1970]["mean"].mean(), 2),"| 1990+ =", round(movies[movies.year >= 1990]["mean"].mean(), 2))


unique rating_year: 2
movies before 1970: 179 of 1677
avg of movie means: before 1970 = 3.58 | 1990+ = 2.98


In [16]:
movie_mean = df.groupby("movie_id").rating.mean().rename("movie_mean")
by_movie = (genre_df.drop_duplicates(["movie_id", "genre"]).merge(movie_mean, on="movie_id").groupby("genre").movie_mean.mean())
by_rating = genre_df.groupby("genre").rating.mean()
cmp = pd.DataFrame({"all_ratings": by_rating, "movie_first": by_movie}).drop("unknown").round(2)
cmp["rank_ratings"] = cmp.all_ratings.rank(ascending=False).astype(int)
cmp["rank_movie"] = cmp.movie_first.rank(ascending=False).astype(int)
cmp.sort_values("rank_ratings")

,all_ratings,movie_first,rank_ratings,rank_movie
genre,,,,
Film-Noir,3.92,3.55,1,1
War,3.82,3.49,2,2
Drama,3.69,3.19,3,9
Documentary,3.67,3.23,4,7
Mystery,3.64,3.34,5,4
Crime,3.63,3.21,6,8
Romance,3.62,3.24,7,6
Western,3.61,3.19,8,9
Animation,3.58,3.30,9,5


## 3.4 What can this data actually tell you?

**Answer at least 3 of these 5**, using your own results as evidence.

**1. Who is in this data?** Check the fewest and the most ratings any single user has given (`df.groupby('user_id').size()`), and look at `rating_year`. What kind of person ends up in a dataset like this, and who is missing? How might that tilt your Q2 answer? *(Week 5: sampling bias)*

**2. How sure are you about the bottom genre?** Take the lowest-rated genre from your Q2. How many ratings *and* how many distinct movies is it built on? Estimate a rough 95% interval: mean ± 1.96 × std / √n. Would you bet it's truly the worst genre? *(Week 5: standard error and confidence intervals. Also ask yourself: are 5,000 ratings of the same handful of movies really 5,000 independent pieces of evidence? Remember the one-row-per-model lesson from last week's homework.)*

**3. What can Q3 really say about time?** How many different values does `rating_year` have? So can this data tell you whether people's taste changed over time? What does your Q3 trend actually describe instead? Why might an old movie in this dataset score higher than a recent one? *(Think: who rates a 1940s film in 1997, and which old movies are even in the data?)*

**4. Does "average" depend on how you average?** Recompute Q2 two ways: (a) average over all ratings, (b) average each movie first, then average the movies within each genre. Does the top genre change? "Which genre do *people* like most?" and "which genre has the best *movies*?" are different questions. *(Week 5: confounding, Simpson's paradox)*

**5. Deployment risk.** A studio exec uses your dashboard to decide which genre to green-light. What's the most misleading conclusion they could draw from one of your charts? Write the one-line caveat you'd put under that chart.

**✏️ Your answers:** _(label each one with its number)_

**1. Who is in this data?**
943 users who each rated at least 20 movies (median 65, max 737), 71% men, median age 31, and `rating_year` is only 1997 or 1998. People who rarely watch movies are missing, so genres like Children, Fantasy or Horror may look worse than they would to everyone.

**2. How sure are you about the bottom genre?**
Fantasy is 3.22 on 1,352 ratings but only 22 movies, and three of them are 46% of the ratings. The 95% interval is 3.16 to 3.28, which overlaps Horror (3.26 to 3.32), so I wouldn't bet it's truly the worst. And since the ratings come from the same 22 movies, the interval is probably too narrow.

**3. What can Q3 really say about time?**
`rating_year` has only 2 values, so it can't show taste changing; the trend just compares release years rated by the same group in 1997-98. Old films may score higher (3.58 for pre-1970 vs 2.98 for 1990+, averaging each movie) because people who choose to watch a classic already like it, and only 179 of the 1,677 movies are from before 1970.

**4. Does "average" depend on how you average?**
Film-Noir is first both ways, but the order changes a lot: Drama goes from 3rd (3.69) to 9th (3.19) when I average each movie first, and Musical jumps from 11th to 3rd. Horror also becomes the lowest (2.73) instead of Fantasy, so "what people like" and "which genre has the best movies" are different questions.

**5. Deployment risk.**
An exec could read the Q2 chart as "green-light Film-Noir, avoid Fantasy". Caveat: *"Based on 1997-98 raters, and the top and bottom genres rest on about 20 movies each with only a 0.7-star gap, so treat this as a hint, not a decision."*




## 3.5 Verdict

**Where I'd override the AI** (something it got wrong, hid, or did poorly, and what your version did better):

**Where the AI held up** (something it did well, or a choice you doubted at first that turned out right):

**One sentence:** looking back at both weeks, who was driving, you or the AI, and when?


**Where I'd override the AI:**
The numbers. The dashboard's results were right, but it quietly dropped the 30 ratings with no release year when loading, so some counts were off by 1 or 2 (Drama 724 vs my 725, Documentary 48 vs 50, 1,676 films vs 1,682). It also flipped Documentary and Drama in Q2 (3.69 vs 3.69, a near tie). I traced the cause by redoing my counts without those rows, and I used the full data where the question allowed it. Its Q2 and Q4 charts also hid sample sizes in hover or left them out, where I showed counts next to each average.

**Where the AI held up:**
The visuals. The dashboard was cleaner, more readable and more informative than my charts. The dot plot around the overall mean, the line with a vote-count strip, and the rank board showing who drops out at the 150-vote floor each answered the question better than my plain bars. Its numbers also matched mine for Q3 and Q4 exactly, which made me trust them more.

**One sentence:** About 70/30: the AI drove the building and the numbers, and I drove the design direction and choices, like banning the default look and picking the concept.


---
## Optional stretch

Only if you want more practice:

- Pick 4 genres. How does the mean rating change as viewer age increases? (Try more than 4 and see which genres make the most interesting chart.)
- Plot number of ratings vs. mean rating for each movie. Do the most-rated movies get higher ratings? Is there a correlation? *(Week 5: correlation vs. causation)*

---
## Before you submit

- [ ] Copy this notebook into the `exercise/` folder and rename it `Firstname_Lastname_week_06_exercise.ipynb`
- [ ] **Kernel → Restart & Run All** and make sure it runs top to bottom with your charts showing
- [ ] Every ✏️ cell and every fill-in line is filled in
- [ ] Open a PR from your fork, same as every other week